In [ ]:
import json
import time
from datetime import date, timedelta
from pathlib import Path

import pandas as pd
import requests

In [ ]:
dbutils.widgets.text("locations_csv_path", "../config/locations.csv", "Locations CSV")
dbutils.widgets.text("output_volume", "/Volumes/workspace/raw_weather/clima_pe", "Output Volume")
dbutils.widgets.text("forecast_days", "16", "Forecast Days")
dbutils.widgets.text("historical_days", "365", "Historical Days")
dbutils.widgets.text("timeout", "30", "Time Out")

In [ ]:
# locations_csv_path asume el notebook corriendo dentro de un Databricks Repo,
# donde el working dir es la carpeta del notebook (A_Raw/../config/locations.csv)
locations_csv_path = dbutils.widgets.get("locations_csv_path")
output_volume = Path(dbutils.widgets.get("output_volume"))
forecast_days = int(dbutils.widgets.get("forecast_days"))
historical_days = int(dbutils.widgets.get("historical_days"))
timeout = int(dbutils.widgets.get("timeout"))

locations = pd.read_csv(locations_csv_path)

In [ ]:
# Crea esquema y volumen si no existen
current_catalog = spark.sql("select current_catalog()").first()[0]
catalog = current_catalog
schema = "raw_weather"
volume = "clima_pe"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

In [ ]:
HOURLY_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "wind_gusts_10m",
    "wind_direction_10m",
    "shortwave_radiation",
    "et0_fao_evapotranspiration",
    "uv_index",
    "weather_code",
]

DAILY_VARS = [
    "weather_code",
    "temperature_2m_max",
    "temperature_2m_min",
    "temperature_2m_mean",
    "precipitation_sum",
    "wind_speed_10m_max",
    "wind_gusts_10m_max",
    "shortwave_radiation_sum",
    "et0_fao_evapotranspiration",
    "uv_index_max",
    "sunshine_duration",
]

In [ ]:
def fetch(url: str, params: dict, retries: int = 5) -> dict:
    # Open-Meteo devuelve 429 si se llama muy seguido para 25 ubicaciones;
    # reintenta respetando Retry-After en vez de fallar la corrida completa.
    for attempt in range(retries):
        response = requests.get(url, params=params, timeout=timeout)
        if response.status_code == 429:
            wait = int(response.headers.get("Retry-After", 10 * (attempt + 1)))
            time.sleep(wait)
            continue
        response.raise_for_status()
        return response.json()
    response.raise_for_status()
    return response.json()


def out_path_for(data_type: str, region: str) -> Path:
    partition_date = date.today().isoformat()
    out_dir = output_volume / data_type / region.replace(" ", "_")
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / f"{partition_date}.json"


def save(payload: dict, out_path: Path) -> Path:
    out_path.write_text(json.dumps(payload), encoding="utf-8")
    return out_path

In [ ]:
today = date.today()
start_historical = (today - timedelta(days=historical_days)).isoformat()
end_historical = (today - timedelta(days=1)).isoformat()

saved_files = []
for row in locations.itertuples():
    # Idempotente: si el archivo del día ya existe, no vuelve a llamar la API
    forecast_path = out_path_for("forecast", row.region)
    if not forecast_path.exists():
        forecast = fetch(
            "https://api.open-meteo.com/v1/forecast",
            {
                "latitude": row.latitude,
                "longitude": row.longitude,
                "hourly": ",".join(HOURLY_VARS),
                "daily": ",".join(DAILY_VARS),
                "forecast_days": forecast_days,
                "timezone": "auto",
            },
        )
        forecast["location_id"] = row.location_id
        forecast["region"] = row.region
        saved_files.append(save(forecast, forecast_path))
        time.sleep(1.0)

    historical_path = out_path_for("historical", row.region)
    if not historical_path.exists():
        historical = fetch(
            "https://archive-api.open-meteo.com/v1/archive",
            {
                "latitude": row.latitude,
                "longitude": row.longitude,
                "start_date": start_historical,
                "end_date": end_historical,
                "hourly": ",".join(HOURLY_VARS),
                "daily": ",".join(DAILY_VARS),
                "timezone": "auto",
            },
        )
        historical["location_id"] = row.location_id
        historical["region"] = row.region
        saved_files.append(save(historical, historical_path))
        time.sleep(1.0)

print(f"Archivos nuevos guardados: {len(saved_files)}")

In [ ]:
# Qué hay en el Volume hasta el momento (no solo lo que se descargó en esta corrida)
resumen = [
    {"data_type": data_type_dir.name, "region": region_dir.name, "archivos": len(list(region_dir.glob("*.json")))}
    for data_type_dir in output_volume.iterdir()
    for region_dir in data_type_dir.iterdir()
]
display(pd.DataFrame(resumen).sort_values(["data_type", "region"], ignore_index=True))